# FPL Bot v1 — driver notebook

Thin driver for the FPL advisory bot. Every algorithm lives in the `bot/` package;
this notebook only wires the modules together and shows the output.

**This bot never executes transfers.** It produces recommendations for a human to act on.

| Section | What it does |
|---|---|
| 1 | Data collection and inspection |
| 2 | Feature engineering (EWMA, team strength, de-vigged odds) |
| 3 | ML model training + feature importances |
| 4 | Monte Carlo simulation of a gameweek |
| 5 | Squad optimisation for 2026/27 |
| 6 | Walk-forward backtest (MAE + Spearman) |
| 7 | RL chip agent |
| 8 | Research summary |

**No API keys are needed anywhere.** Every source is a public, unauthenticated HTTP GET.

## Cell 0 — Colab setup

Skip this cell if you are running locally from a clone of the repo.

In [ ]:
# --- Google Colab setup -------------------------------------------------
# Safe to re-run. Skip entirely when running locally.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("fpl-auto"):
        subprocess.run(["git", "clone",
                        "https://github.com/DH4410/fpl-auto.git"], check=True)
    os.chdir("fpl-auto")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements_bot.txt"], check=True)
else:
    # Running locally: make sure the repo root (the parent of bot/) is importable.
    repo_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" \
        else os.getcwd()
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    os.chdir(repo_root)

print("working directory:", os.getcwd())
print("bot package present:", os.path.isdir("bot"))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bot import data_collector as dc
from bot import feature_engineering as fe
from bot import models as M
from bot import simulator as S
from bot import optimizer as O
from bot import rl_agent as R
from bot.fpl_rules import RULES, GKP, DEF, MID, FWD, POSITION_NAMES

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("modules loaded")

---
## 1. Data collection and inspection

Sources, all free and unauthenticated:

* **Official FPL API** — `bootstrap-static`, `fixtures`, `element-summary`, `event/{gw}/live`,
  `team/set-piece-notes`. Primary source for the current player pool.
* **Vaastav's GitHub CSVs** — historical per-gameweek data for training (2024-25, 2025-26).
* **football-data.co.uk** — free historical CSVs with closing bookmaker odds.

Everything is cached to `bot/cache/` and auto-refreshes after 6 hours.

In [ ]:
bs = dc.bootstrap_frames()
players, teams, events = bs["players"], bs["teams"], bs["events"]

print(f"players: {len(players)}   teams: {len(teams)}   gameweeks: {len(events)}")
print("next gameweek:", dc.current_gameweek())

pool = dc.build_player_pool()
pool[["web_name", "team_short", "element_type", "price", "status",
      "total_points", "selected_by_percent"]].head(10)

In [ ]:
# Historical training data. NOTE the defensive-contribution caveat:
# 2024-25 predates the DC rule and has none of those columns.
history = dc.load_multi_season_history(("2024-25", "2025-26"))
print("rows:", len(history))
for season, cov in history.attrs["coverage"].items():
    print(f"  {season}: {cov['rows']:>6} rows | "
          f"DC columns present: {cov['has_defcon'] or 'NONE'}")
print("\nseasons usable for the defensive-contribution model:",
      history.attrs["defcon_seasons"])

In [ ]:
# Free historical odds (Pinnacle closing prices) — no API key.
odds_2425 = dc.fetch_football_data_odds("2024-25")
print("odds source columns used:", odds_2425.attrs["odds_source"])
odds_2425[["date", "HomeTeam", "AwayTeam", "home_goals", "away_goals",
           "odds_home", "odds_draw", "odds_away", "odds_over25"]].head()

---
## 2. Feature engineering

Four families of features:

1. **EWMA form** — recency-weighted rolling stats, shifted one gameweek so no label leaks in.
2. **Dixon-Coles team strength** — attack/defence ratings fitted by maximum likelihood.
3. **De-vigged odds** — Shin's method strips the bookmaker margin; the fair probabilities
   are inverted into implied goal rates.
4. **Promoted-team priors** — newly promoted clubs get worst-quartile ratings, not league-average.

In [ ]:
feat = fe.compute_ewma_stats(history, alpha=0.25)
feat = fe.rolling_window_stats(feat)

# Vaastav stores position as text; the models want the numeric element_type.
feat["element_type"] = feat["position"].map(
    {"GK": GKP, "GKP": GKP, "DEF": DEF, "MID": MID, "FWD": FWD})
for pid, label in ((GKP, "gkp"), (DEF, "def"), (MID, "mid"), (FWD, "fwd")):
    feat[f"pos_{label}"] = (feat["element_type"] == pid).astype(float)

feature_cols = fe.feature_columns(feat)
print(f"{len(feature_cols)} model features:")
print(feature_cols)

# Leakage check: a player's very first appearance must have no prior form.
first_rows = feat.groupby("element").head(1)
print("\nfirst-appearance rows with NaN EWMA (should be 1.00):",
      round(first_rows["ewma_minutes"].isna().mean(), 2))

In [ ]:
# EWMA form for a few high-profile players.
top = (history.groupby(["element", "name"])["total_points"].sum()
       .sort_values(ascending=False).head(4).reset_index())

fig, ax = plt.subplots(figsize=(11, 4.5))
for _, row in top.iterrows():
    sub = feat[(feat["element"] == row["element"]) &
               (feat["season"] == "2025-26")].sort_values("round")
    if len(sub) > 3:
        ax.plot(sub["round"], sub["ewma_total_points"], marker="o",
                markersize=3, label=row["name"])
ax.set_xlabel("Gameweek")
ax.set_ylabel("EWMA points (alpha=0.25)")
ax.set_title("Recency-weighted form, 2025-26")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Dixon-Coles ratings. Fit on club NAMES, never on FPL team ids:
# FPL reassigns ids alphabetically every season, so id 9 is a different club year to year.
strength = fe.fit_team_strength_by_name(odds_2425, teams)
strength = fe.promoted_team_priors(teams, strength, quantile=0.25)

print(f"home advantage: {strength.attrs.get('home_advantage', float('nan')):.3f}")
print(f"Dixon-Coles rho: {strength.attrs.get('rho', float('nan')):.3f}")
print(f"promoted/unseen clubs seeded with priors: {int(strength['is_prior'].sum())}\n")

strength.sort_values("attack", ascending=False)[
    ["team_name", "attack", "defence", "expected_scored",
     "expected_conceded", "is_prior"]].round(3)

In [ ]:
# Team strength matrix as a heatmap.
plot_df = strength.dropna(subset=["team_name"]).sort_values("attack", ascending=False)
fig, ax = plt.subplots(figsize=(7, 7))
mat = plot_df[["attack", "defence"]].to_numpy()
im = ax.imshow(mat, cmap="RdYlGn", aspect="auto")
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["team_name"], fontsize=8)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Attack", "Defence"])
ax.set_title("Dixon-Coles team ratings (higher = better)")
plt.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# De-vigged odds -> implied goal rates. Shin's method shifts probability
# toward the favourite relative to naive proportional de-vigging.
sample = odds_2425[["odds_home", "odds_draw", "odds_away"]].iloc[0].to_numpy()
print("raw odds          :", sample)
print("implied (1/odds)  :", (1 / sample).round(4), "sum =", round((1 / sample).sum(), 4))
print("proportional      :", fe.devig_odds(sample, method="proportional").round(4))
print("Shin              :", fe.devig_odds(sample, method="shin").round(4))

market = fe.odds_features(odds_2425.head(20), teams)
market[["HomeTeam", "AwayTeam", "p_home", "p_draw", "p_away",
        "lambda_home", "lambda_away", "mkt_clean_sheet_home"]].round(3).head()

In [ ]:
# Free keyword news sentiment — no LLM, no API key.
avail = fe.news_sentiment_features(pool, dc.fetch_set_piece_notes())
flagged = avail[avail["news"].astype(str).str.len() > 0]
print(f"{len(flagged)} players currently carry an FPL news note\n")
flagged.nsmallest(10, "availability_index")[
    ["web_name", "team_short", "news", "news_sentiment",
     "avail_next", "availability_index"]]

---
## 3. ML model training

Four sub-models feed one predictor. Rather than regressing points directly, we predict the
*components* and compose them through the exact FPL rules.

| Model | Target | Note |
|---|---|---|
| `MinutesModel` | minutes 0-90 | asymmetric loss — over-prediction penalised 3x |
| `AttackModel` | xG/90 and xA/90 | two LightGBM regressors |
| `DefenseModel` | clean sheet prob + DC/90 | **DC head trains on 2025-26 only** |
| `BonusModel` | BPS per appearance | plus an empirical BPS→bonus curve |

**On defensive contribution:** 2024-25's `merged_gw.csv` contains no
`clearances_blocks_interceptions`, `recoveries`, `tackles` or `defensive_contribution`
columns — the statistic did not exist that season. The DC head therefore trains only on
rows that genuinely carry it, and falls back to positional priors otherwise. It never
zero-fills, since zero would assert "made no clearances" rather than "unknown".

In [ ]:
targets = M.build_training_targets(feat)

X_all = feat[feature_cols].astype(float)
X_all = X_all.fillna(X_all.median())

valid = targets["minutes"].notna().to_numpy()
X = X_all[valid].reset_index(drop=True)
y = {k: pd.Series(np.asarray(v))[valid].reset_index(drop=True)
     for k, v in targets.items()}
element_types = feat["element_type"].to_numpy()[valid]

print(f"training rows: {len(X)}")
print(f"rows with defensive-contribution labels: {int(y['defcon_per90'].notna().sum())}")

In [ ]:
predictor = M.FPLPointsPredictor()

predictor.minutes_model.train(X, y["minutes"])
print("minutes model trained")

rate_rows = y["xg_per90"].notna().to_numpy()
predictor.attack_model.train(X[rate_rows], y["xg_per90"][rate_rows],
                             y["xa_per90"][rate_rows])
print("attack model trained")

predictor.defense_model.calibrate_position_priors(feat)
predictor.defense_model.train(X, y["clean_sheet"], y["defcon_per90"])
print("defense model trained | DC head is rule-based fallback:",
      predictor.defense_model.defcon_is_rule_based)

predictor.bonus_model.train(X, y["bps"], y["bonus"])
print("bonus model trained")

In [ ]:
# Feature importances.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (title, model) in zip(axes, [
        ("Minutes", predictor.minutes_model),
        ("Attack (xG)", predictor.attack_model),
        ("Bonus (BPS)", predictor.bonus_model)]):
    imp = model.feature_importances(10)
    ax.barh(imp["feature"][::-1], imp["importance"][::-1])
    ax.set_title(f"{title} — top features")
    ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Component-wise expected points, so a projection can be audited.
proj = predictor.predict(X.head(3000), element_types[:3000])
proj.describe().round(3)[
    ["expected_points", "expected_minutes", "p60", "clean_sheet_prob",
     "p_defcon", "expected_bonus"]]

---
## 4. Monte Carlo simulation

The predictor gives an expectation; decisions need the whole distribution. Captaincy is a
bet on the right tail, and a differential is only worth owning if its upside is fat.

Scorelines come from a bivariate Poisson; team goals are then split among players by a
vectorised multinomial that conserves the team total exactly.

In [ ]:
strength_sim = strength.copy()
match_sim = S.BivariatePoissonSimulator(strength=strength_sim)

# Fit the shared-covariance term on real scorelines.
name_map = dc.match_team_names(
    pd.unique(pd.concat([odds_2425["HomeTeam"], odds_2425["AwayTeam"]])), teams)
results = odds_2425.rename(columns={"HomeTeam": "home_team", "AwayTeam": "away_team"}).copy()
results["home_team"] = results["home_team"].map(name_map)
results["away_team"] = results["away_team"].map(name_map)
results = results.dropna(subset=["home_team", "away_team"])

match_sim.fit(results)
print(f"lambda3 = {match_sim.lambda3:.4f}   home advantage = {match_sim.home_advantage:.3f}")
print("\nNote: lambda3 ~ 0 is the correct empirical result. Real scorelines are mildly")
print("NEGATIVELY dependent, and a bivariate Poisson can only express positive")
print("correlation, so MLE drives the shared term to zero. The Dixon-Coles tau")
print("correction in the ratings fit handles the low-score dependence instead.")

In [ ]:
# Build the per-player rate table the simulator consumes, from the trained models.
current = pool[pool["available"]].copy().reset_index(drop=True)

hist_features = (feat.sort_values(["season", "round"])
                 .groupby("element").tail(1).set_index("element"))
aligned = hist_features.reindex(current["id"])
X_now = aligned[feature_cols].astype(float)
X_now = X_now.fillna(X_all.median()).reset_index(drop=True)

live = predictor.predict(X_now, current["element_type"].to_numpy())

squad_stats = pd.DataFrame({
    "element": current["id"].to_numpy(),
    "team": current["team"].to_numpy(),
    "element_type": current["element_type"].to_numpy(),
    "xg_per90": live["xg_per90"].to_numpy(),
    "xa_per90": live["xa_per90"].to_numpy(),
    "expected_minutes": live["expected_minutes"].to_numpy(),
    "p60": live["p60"].to_numpy(),
    "defcon_per90": live["defcon_per90"].to_numpy(),
})
squad_stats.head()

In [ ]:
fixtures = dc.fixtures_frame()
gw = dc.current_gameweek() or 1
gw_fixtures = fixtures[fixtures["event"] == gw]
print(f"simulating GW{gw} — {len(gw_fixtures)} fixtures, {len(squad_stats)} players")

season_sim = S.SeasonSimulator(match_sim=match_sim)
summary, samples = season_sim.simulate_gameweek(
    gw_fixtures, squad_stats, n_sims=20_000, return_samples=True)

named = summary.join(pool.set_index("id")[["web_name", "team_short", "price"]])
named.sort_values("mean", ascending=False).head(15)[
    ["web_name", "team_short", "price", "mean", "std", "p80", "p90",
     "p_haul", "p_blank"]].round(2)

In [ ]:
# Distribution shape for the leading captaincy candidates.
top_ids = named.sort_values("mean", ascending=False).head(4).index.tolist()
positions = {e: i for i, e in enumerate(squad_stats["element"])}

fig, ax = plt.subplots(figsize=(11, 4.5))
for element in top_ids:
    draws = samples[positions[element]]
    ax.hist(draws, bins=np.arange(0, 26) - 0.5, alpha=0.45, density=True,
            label=f"{named.loc[element, 'web_name']} (mean {draws.mean():.1f})")
ax.set_xlabel("Points in the gameweek")
ax.set_ylabel("Probability")
ax.set_title("Simulated point distributions — captaincy candidates")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Mean alone is the wrong captaincy criterion — the armband doubles the score,")
print("so upside and blank risk matter more than they do for an ordinary pick.\n")
season_sim.captaincy_analysis(
    summary, samples, squad_stats["element"].to_numpy(), top_n=8).round(3)

---
## 5. Squad optimisation for 2026/27

A mixed-integer program, solved with HiGHS where available and CBC otherwise.
**Gurobi is never used** — it is commercial.

Constraints: £100.0m budget, 2/5/5/3 by position, max 3 per club, a legal formation,
and — for transfers — the 50%-profit selling-price rule.

In [ ]:
opt = O.SquadOptimizer()
print("solver:", type(O.get_solver()).__name__)

candidates = current.copy()
candidates["element"] = candidates["id"]
candidates["expected_points"] = summary["mean"].reindex(candidates["id"]).to_numpy()
candidates["expected_points"] = candidates["expected_points"].fillna(0)

best = opt.optimize_initial_squad(candidates, budget=100.0)

print(f"\ncost: £{best['total_cost']:.1f}m   expected XI points: {best['expected_points']:.1f}")
print(f"captain: {best['captain_name']}   vice: {best['vice_captain_name']}\n")

squad_df = pd.DataFrame(best["squad"])
squad_df["pos"] = squad_df["position"].map(POSITION_NAMES)
squad_df["role"] = np.where(
    squad_df["element"].isin([p["element"] for p in best["starting_xi"]]), "XI", "bench")
squad_df.sort_values(["position", "expected_points"], ascending=[True, False])[
    ["name", "pos", "price", "expected_points", "role"]]

In [ ]:
# Independently verify the recommendation against the rules module.
check = squad_df.rename(columns={"position": "element_type"}).copy()
check["now_cost"] = (check["price"] * 10).round().astype(int)

valid, reason = RULES.validate_squad(check.to_dict("records"), budget=100.0)
print("squad legal:", valid, "|", reason)

xi_positions = [{"element_type": p["position"]} for p in best["starting_xi"]]
print("formation legal:", RULES.validate_formation(xi_positions))
print("max players from one club:", check["team"].value_counts().max())
print("squad composition:", check["element_type"].map(POSITION_NAMES).value_counts().to_dict())

In [ ]:
# Stochastic version: optimise over the Monte Carlo scenarios rather than point estimates.
# This matters because FPL scoring has thresholds (60 minutes, the DC cutoff),
# and E[f(x)] != f(E[x]) when f has steps in it.
order = {e: i for i, e in enumerate(squad_stats["element"])}
rows = [order[e] for e in candidates["id"] if e in order]
scenario_matrix = samples[rows]
stochastic_pool = candidates[candidates["id"].isin(order)].reset_index(drop=True)

risky = opt.optimize_squad_stochastic(stochastic_pool, scenario_matrix,
                                      budget=100.0, risk_aversion=0.0)
safe = opt.optimize_squad_stochastic(stochastic_pool, scenario_matrix,
                                     budget=100.0, risk_aversion=0.5)

overlap = len({p["element"] for p in risky["squad"]} &
              {p["element"] for p in safe["squad"]})
print(f"risk-neutral cost £{risky['total_cost']:.1f}m | "
      f"risk-averse cost £{safe['total_cost']:.1f}m")
print(f"players in common between the two squads: {overlap}/15")

In [ ]:
# Transfer recommendation, demonstrating the selling-price rule.
# A player bought at £5.0m and now worth £5.5m sells for £5.2m, NOT £5.5m.
print("selling price checks (values in £0.1m):")
for paid, now in [(50, 55), (50, 54), (50, 47), (50, 50)]:
    got = RULES.calc_selling_price_tenths(paid, now)
    print(f"  paid {paid}, now {now} -> sells for {got}")

owned = squad_df.rename(columns={"position": "element_type"}).copy()
owned["now_cost"] = (owned["price"] * 10).round().astype(int)
owned["purchase_price"] = owned["now_cost"] - 3   # pretend each player rose £0.3m

rec = opt.optimize_transfers(owned, candidates, free_transfers=1,
                             bank=0.5, n_gw_horizon=3, max_transfers=2)

print(f"\ntransfers: {rec['n_transfers']}   hits: {rec['hits']} "
      f"(-{rec['hit_cost']} pts)   bank after: £{rec['bank_after']:.1f}m")
for t in rec["transfers_out"]:
    print(f"  OUT  {t['name']:<20} sells for £{t['selling_price']:.1f}m")
for t in rec["transfers_in"]:
    print(f"  IN   {t['name']:<20} costs     £{t['cost']:.1f}m")
print("\nadvisory only — nothing was executed:", rec["advisory_only"])

---
## 6. Backtest — walk-forward validation

At each gameweek *t* the model trains on everything strictly before *t* and is scored on
*t* alone. Ordinary k-fold cross-validation is invalid here: shuffling rows lets the model
learn from GW30 to predict GW12, which inflates every metric.

**Spearman rank correlation matters more than MAE.** The optimiser only needs players
ordered correctly, not their totals nailed exactly.

In [ ]:
backtest_frame = feat.copy()
for col in feature_cols:
    backtest_frame[col] = X_all[col]

wf_minutes = M.walk_forward_validation(
    backtest_frame, feature_cols, "minutes",
    lambda: M.MinutesModel(n_estimators=120),
    min_train_periods=6, step=3)

print("MINUTES MODEL — walk-forward")
print(f"  mean MAE      : {wf_minutes['mae'].mean():.2f} minutes")
print(f"  mean Spearman : {wf_minutes['spearman'].mean():.3f}")
print(f"  mean bias     : {wf_minutes['bias'].mean():.2f}  (negative = conservative, as designed)")
wf_minutes.round(3)

In [ ]:
wf_points = M.walk_forward_validation(
    backtest_frame, feature_cols, "total_points",
    lambda: M.BonusModel(n_estimators=120),
    min_train_periods=6, step=3)

print("POINTS — walk-forward")
print(f"  mean MAE      : {wf_points['mae'].mean():.3f} points")
print(f"  mean Spearman : {wf_points['spearman'].mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(len(wf_minutes)), wf_minutes["mae"], marker="o", label="minutes MAE")
axes[0].set_title("Minutes MAE by evaluation period")
axes[0].set_xlabel("period"); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(range(len(wf_minutes)), wf_minutes["spearman"], marker="o", label="minutes")
axes[1].plot(range(len(wf_points)), wf_points["spearman"], marker="s", label="points")
axes[1].set_title("Spearman rank correlation")
axes[1].set_xlabel("period"); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Record the headline numbers so they can be pasted into
# research/starting_point/README.md.
results = {
    "minutes_mae": round(float(wf_minutes["mae"].mean()), 3),
    "minutes_spearman": round(float(wf_minutes["spearman"].mean()), 3),
    "minutes_bias": round(float(wf_minutes["bias"].mean()), 3),
    "points_mae": round(float(wf_points["mae"].mean()), 3),
    "points_spearman": round(float(wf_points["spearman"].mean()), 3),
    "n_periods": int(len(wf_minutes)),
    "training_rows": int(len(X)),
    "n_features": len(feature_cols),
}
for k, v in results.items():
    print(f"{k:>18} : {v}")

---
## 7. RL chip agent

Chip timing is a sequencing problem under uncertainty: spending Bench Boost in GW7 forecloses
the double gameweek in GW34. That is a Markov decision process, so it gets reinforcement
learning rather than a myopic rule.

Set `N_EPISODES` low for a quick smoke test and high for a real run.

In [ ]:
N_EPISODES = 300   # raise to 5000+ for a serious run

replay = R.SeasonReplayGenerator(history_df=history)
env = R.FPLEnv(replay=replay, seed=0)
print("observation:", env.observation_space.shape, " actions:", env.action_space.n)
print("action meanings:", R.ACTIONS)

# Benchmark 1: never play a chip. Benchmark 2: the hand-written heuristic.
def run_policy(policy, n=200):
    totals = []
    for ep in range(n):
        obs, _ = env.reset(seed=5000 + ep)
        done, total = False, 0.0
        while not done:
            obs, reward, done, _, _ = env.step(policy(env))
            total += reward
        totals.append(total)
    return float(np.mean(totals))

def heuristic(e):
    state = {"gameweek": e.gameweek, "chips_available": e.chips_available,
             "n_fixtures": e.season["n_fixtures"][e.gameweek - 1]}
    chip = R.baseline_chip_policy(state)
    return R.ACTIONS.index(chip) if chip else 0

no_chips = run_policy(lambda e: 0)
baseline = run_policy(heuristic)
print(f"\nno chips at all : {no_chips:.0f} pts/season")
print(f"heuristic policy: {baseline:.0f} pts/season  (+{baseline - no_chips:.0f})")
print("\nAn RL agent that cannot beat the heuristic is not worth deploying.")

In [ ]:
# Training needs stable-baselines3 (which pulls in torch).
try:
    agent = R.FPLChipAgent(replay=replay)
    agent.train(n_episodes=N_EPISODES)

    log = agent.evaluate(n_episodes=100)
    seasons = log[log["chip"] == "_total"]["reward_gain"]
    print(f"\nPPO agent      : {seasons.mean():.0f} pts/season")
    print(f"vs heuristic   : {seasons.mean() - baseline:+.0f}")
    print(f"vs no chips    : {seasons.mean() - no_chips:+.0f}")

    timing = log[log["chip"] != "_total"].groupby("chip")["gameweek"].agg(
        ["mean", "std", "count"])
    print("\nWhen the agent plays each chip:")
    display(timing.round(1))

    agent.save("bot/cache/chip_agent")
except ImportError as exc:
    print("stable-baselines3 not installed, skipping RL training:", exc)
    print("install with: pip install stable-baselines3")

---
## 8. Research summary

In [ ]:
summary = f'''
FPL BOT v1 -- SUMMARY
==============================================================

TECHNIQUES IMPLEMENTED
  Module 1  Feature engineering
            - EWMA form (alpha=0.25), leak-safe one-gameweek shift
            - Dixon-Coles attack/defence ratings, MLE with time decay
            - Shin de-vigging of free football-data.co.uk closing odds
            - Market-implied goal rates by Poisson inversion
            - Promoted-team worst-quartile priors
            - Keyword news sentiment (free, no LLM service)
  Module 2  Four ML sub-models
            - Minutes with asymmetric loss (over-prediction penalised 3x)
            - Attack: xG/90 and xA/90 (LightGBM)
            - Defense: clean-sheet classifier + DC/90 regressor
            - Bonus: BPS regressor + empirical BPS->bonus curve
  Module 3  Bivariate Poisson simulation, vectorised multinomial attribution
  Module 4  MIP optimisation (HiGHS/CBC) for squad, XI, transfers and chips
  Module 5  PPO chip-timing agent over a Gymnasium season environment

BACKTEST (walk-forward, expanding window)
  training rows      : {results["training_rows"]}
  features           : {results["n_features"]}
  evaluation periods : {results["n_periods"]}
  minutes MAE        : {results["minutes_mae"]}
  minutes Spearman   : {results["minutes_spearman"]}
  points Spearman    : {results["points_spearman"]}

DEFERRED, AND WHY
  Paid odds APIs     : not used at all. Free football-data.co.uk closing
                       odds (Pinnacle) replace them entirely.
  LLM news sentiment : needs an API key. Replaced by a domain-specific
                       keyword scorer, which handles football injury text
                       better than general-purpose sentiment anyway.
  Effective ownership: needs top-10k pick scraping. Belongs in the
                       optimiser objective (rank, not points).
  Shot-level xG      : the free `understat` package would supply it;
                       aggregate xG from the FPL API covers current needs.

KNOWN LIMITATIONS
  - No 2026/27 gameweek data exists yet, so the models are trained on
    2024-25 and 2025-26 and applied out of sample to a new season.
  - Defensive contribution has ONE season of history (2025-26).
  - Bonus is approximated from a player own involvement rather than
    simulating all 22 players BPS.
  - The RL agent trains on simulated seasons, so it inherits every bias
    in the simulator.

THIS BOT NEVER EXECUTES TRANSFERS. Recommendations only.
'''
print(summary)